In [73]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [93]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_2016_2022_tcdf.csv")

In [94]:
df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

In [95]:
df_lt

,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,...,LBGD,PC,ARP,ROL,GE,COC,RQ,SSB,PT,EUPC
0,-1.325279,-0.103863,1.411254,-1.634729,-0.396320,-0.564076,2.157277,0.255824,0.527984,-0.636154,...,0.315077,-1.140150,0.432038,0.138039,0.838054,8.219949e-02,-0.572892,-1.439853e+00,-1.417170e+00,-1.269104
1,-0.893231,0.802577,0.744734,-0.976133,-1.358483,-0.564076,-0.588348,-0.190922,-0.158045,-0.367987,...,0.596791,-1.023870,1.377122,-0.634981,-2.249514,-1.232992e+00,-0.196932,-9.354275e-01,-1.404279e+00,-0.988911
2,-0.446529,0.758384,0.411474,0.505707,0.325303,0.564076,-0.588348,-1.004907,0.930366,-1.213405,...,0.708389,-0.326189,1.377122,-1.408002,0.838054,-1.643990e+00,-0.948852,-2.860967e-01,4.524547e-01,0.015073
3,-0.065089,0.392757,-0.350263,0.835005,1.407737,2.256304,0.784465,-0.728531,1.207416,-0.723025,...,0.720975,0.117790,-0.796570,-0.055216,0.220541,9.125977e-16,-0.196932,8.678250e-01,1.117978e+00,-0.724118
4,0.050573,-1.820490,-1.469065,-0.399862,0.926655,-0.564076,-0.588348,2.190459,0.396055,0.094695,...,-2.344221,0.234071,-0.513045,-0.828236,0.220541,9.041944e-01,-1.074172,1.793552e+00,1.251016e+00,1.718146
5,0.769873,1.057946,-1.302435,0.176410,-1.358483,-0.564076,-0.588348,0.199034,-1.451888,0.972217,...,-0.115920,0.022652,-1.363620,1.490826,0.529297,1.232992e+00,1.306910,1.342070e-13,-1.103769e-14,0.902789
6,1.909682,-1.087311,0.554300,1.493601,0.453591,-0.564076,-0.588348,-0.720959,-1.451888,1.873660,...,0.118909,2.115696,-0.513045,1.297570,-0.396973,6.575959e-01,1.682870,1.342070e-13,-1.103769e-14,0.346125


In [96]:
# lt = pd.DataFrame()
# lt = df_lt.copy()
# # lt['EUPC_1'] = lt['EUPC'].shift(1)
# # lt['GDPCG_3'] = lt['GDPCG'].shift(3)
# lt['EUPC_2'] = lt['EUPC'].shift(2)
# # lt['QOR_3'] = lt['QOR'].shift(3)

In [97]:
lt = lt.dropna()

In [98]:
sm = StructureModel()
sm_1 = StructureModel()
sm_2 = StructureModel()

In [99]:
sm.add_edges_from([
     ('EUPC', 'QOR')
])

sm_1.add_edges_from([
     ('EUPC', 'LBGD')
])

In [100]:
sm.edges
sm_1.edges


OutEdgeView([('EUPC', 'LBGD')])

In [101]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lt_2016_2022.html")

Graphs/fully_connected_lt_2016_2022.html


In [102]:
bn = BayesianNetwork(sm)
bn_1 = BayesianNetwork(sm_1)

In [103]:
discretised_lt = pd.DataFrame(index=lt.index)

for col in lt.columns:
    no_unique = lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lt)

Discretised data:
   GDPC  GDPCG  VABI  EMP  PAG  QOA  QOR  YUA  GGAG  REST  ...  PC  ARP  ROL  \
2     0      2     2    1    0    1    0    0     2     0  ...   0    2    0   
3     0      1     1    2    2    1    0    0     2     0  ...   1    0    1   
4     1      0     0    0    2    0    0    2     1     1  ...   2    1    0   
5     2      2     0    0    0    0    0    2     0     2  ...   0    0    2   
6     2      0     2    2    1    0    0    1     0     2  ...   2    1    2   

   GE  COC  RQ  SSB  PT  EUPC  EUPC_2  
2   2    0   0    0   1     0       0  
3   0    0   1    2   2     0       0  
4   0    2   0    2   2     2       2  
5   2    2   2    0   0     2       1  
6   0    1   2    0   0     1       2  

[5 rows x 25 columns]


In [104]:
discretised_lt = discretised_lt.reset_index(drop=True)
bn.fit_node_states(discretised_lt)
baseline_auc = utils.get_avg_auc_all_info(discretised_lt, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.547776222229004 seconds
Processing fold 1 using 7 cores takes 2.926353931427002 seconds
Processing fold 2 using 7 cores takes 2.91749906539917 seconds
Processing fold 3 using 7 cores takes 2.9518649578094482 seconds
Processing fold 4 using 7 cores takes 2.933594226837158 seconds
Baseline AUC: nan


In [105]:
discretised_lt = discretised_lt.reset_index(drop=True)
bn_1.fit_node_states(discretised_lt)
baseline_auc = utils.get_avg_auc_all_info(discretised_lt, bn_1, n_splits=4)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 2.9861090183258057 seconds
Processing fold 1 using 7 cores takes 2.9625771045684814 seconds
Processing fold 2 using 7 cores takes 2.9607908725738525 seconds
Processing fold 3 using 7 cores takes 2.9225778579711914 seconds
Baseline AUC: 0.8125


In [106]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'LBGD')]
edges_to_remove = [('EUPC', 'LBGD')]

bn_with_lv = copy.deepcopy(bn_1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [107]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_2016_2022_2.html")

Graphs/node_added_lt_2016_2022_2.html


In [108]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and LBGD': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.123244047164917 seconds
Processing fold 1 using 7 cores takes 3.172041893005371 seconds
Processing fold 2 using 7 cores takes 3.064110040664673 seconds
Processing fold 3 using 7 cores takes 3.446802854537964 seconds
Processing fold 4 using 7 cores takes 3.0877890586853027 seconds
AUC from adding LV between 'EUPC' and LBGD': 0.9
